In [0]:
%pip install -U databricks-vectorsearch transformers accelerate sentence-transformers langgraph
dbutils.library.restartPython()

In [0]:
from typing import List, Dict, TypedDict, Optional
import re
import json
from databricks.vector_search.client import VectorSearchClient
from transformers import pipeline

In [0]:
VECTOR_SEARCH_ENDPOINT = "legal_rag_endpoint"
LEGAL_INDEX_NAME = "workspace.default.legal_docs_vector_index"
SCHEME_INDEX_NAME = "workspace.default.legal_docs_vector_index"
IPC_BNS_INDEX_NAME = "workspace.default.legal_docs_vector_index"
SUMMARY_INDEX_NAME = "workspace.default.legal_docs_vector_index"
RETURN_COLUMNS = ["chunk_id", "title", "page", "source_file", "domain", "text"]

In [0]:
vsc = VectorSearchClient()

legal_index = vsc.get_index(
    endpoint_name=VECTOR_SEARCH_ENDPOINT,
    index_name=LEGAL_INDEX_NAME,
)

scheme_index = vsc.get_index(
    endpoint_name=VECTOR_SEARCH_ENDPOINT,
    index_name=SCHEME_INDEX_NAME,
)

ipc_bns_index = vsc.get_index(
    endpoint_name=VECTOR_SEARCH_ENDPOINT,
    index_name=IPC_BNS_INDEX_NAME,
)

summary_index = vsc.get_index(
    endpoint_name=VECTOR_SEARCH_ENDPOINT,
    index_name=SUMMARY_INDEX_NAME,
)

In [0]:
generator = pipeline(
    "text2text-generation",
    model="google/flan-t5-base",
    device=-1
)

In [0]:
def extract_records(vs_response) -> List[Dict]:
    """
    Convert Vector Search response into a list of dictionaries.
    Handles common SDK response shapes.
    """
    if hasattr(vs_response, "to_dict"):
        vs_response = vs_response.to_dict()

    if isinstance(vs_response, list):
        return vs_response

    if isinstance(vs_response, dict):
        # Common shape: {"result": {"data_array": [...], "manifest": {"columns": [...]}}}
        result = vs_response.get("result", vs_response)

        if isinstance(result, dict):
            data_array = result.get("data_array", [])
            manifest = result.get("manifest", {})
            cols = []

            if isinstance(manifest, dict):
                raw_cols = manifest.get("columns", [])
                for c in raw_cols:
                    if isinstance(c, dict):
                        cols.append(c.get("name"))
                    else:
                        cols.append(c)

            records = []
            for row in data_array:
                if isinstance(row, dict):
                    records.append(row)
                elif isinstance(row, (list, tuple)) and cols:
                    records.append(dict(zip(cols, row)))
                else:
                    records.append({"value": row})
            return records

    return []

In [0]:
def build_context_block(chunks: List[Dict]) -> str:
    parts = []
    for i, c in enumerate(chunks, start=1):
        header = (
            f"[{i}] {c.get('title', 'Unknown')} | "
            f"page {c.get('page', 'NA')} | "
            f"{c.get('source_file', 'NA')}"
        )
        text = c.get("text", "")
        parts.append(f"{header}\n{text}")
    return "\n\n".join(parts)

In [0]:
def build_prompt(query: str, chunks: List[Dict], task: str) -> str:
    context = build_context_block(chunks)

    task_map = {
        "legal_qa": "Answer the legal question using only the provided context.",
        "scheme_query": "Explain the relevant government schemes using only the provided context.",
        "ipc_bns_comparison": "Compare IPC and BNS using only the provided context.",
        "legal_summarization": "Summarize the legal text simply using only the provided context.",
    }

    instructions = task_map.get(task, task_map["legal_qa"])

    return f"""
You are Nyaya-Sahayak, an Indian legal and public welfare assistant.

Rules:
1. Use only the provided context.
2. Do not invent section numbers, scheme names, punishments, or eligibility rules.
3. If the context is not enough, say so clearly.
4. Mention source labels like [1], [2] where useful.
5. Keep the answer concise, practical, and grounded.

Task:
{instructions}

Question:
{query}

Context:
{context}

Answer:
""".strip()

In [0]:
def generate_grounded_answer(query: str, chunks: List[Dict], task: str) -> str:
    if not chunks:
        return "I could not find enough direct evidence in the indexed documents to answer this safely."

    prompt = build_prompt(query, chunks, task)

    output = generator(
        prompt,
        max_new_tokens=220,
        do_sample=False,
        temperature=0.0
    )[0]["generated_text"].strip()

    return output

In [0]:
def search_index(index, query: str, k: int = 5, use_hybrid: bool = True) -> List[Dict]:
    resp = index.similarity_search(
        query_text=query,
        columns=RETURN_COLUMNS,
        num_results=k,
        query_type="hybrid" if use_hybrid else "ann",
    )
    return extract_records(resp)

In [0]:
def retrieve_legal(query: str, k: int = 5) -> List[Dict]:
    return search_index(legal_index, query, k=k, use_hybrid=True)

def retrieve_scheme(query: str, k: int = 5) -> List[Dict]:
    return search_index(scheme_index, query, k=k, use_hybrid=True)

def retrieve_ipc_bns(query: str, k: int = 6) -> List[Dict]:
    expanded = f"{query} IPC BNS comparison"
    return search_index(ipc_bns_index, expanded, k=k, use_hybrid=True)

def retrieve_summary(query: str, k: int = 5) -> List[Dict]:
    return search_index(summary_index, query, k=k, use_hybrid=True)

In [0]:
def dedupe_sources(chunks: List[Dict]) -> List[Dict]:
    seen = set()
    out = []
    for c in chunks:
        key = (
            c.get("chunk_id"),
            c.get("source_file"),
            c.get("page"),
            c.get("title"),
        )
        if key not in seen:
            seen.add(key)
            out.append(c)
    return out

In [0]:
def build_citizen_action_pack(intents: List[str], query: str) -> str:
    blocks = []

    if "legal_qa" in intents or "ipc_bns_comparison" in intents:
        blocks.append(
            "### Citizen Action Pack\n"
            "1. Preserve evidence: screenshots, transaction IDs, messages, emails.\n"
            "2. Note the exact time, date, and platform involved.\n"
            "3. File the complaint with the relevant authority or portal.\n"
            "4. Keep a written timeline of events and all acknowledgements."
        )

    if "scheme_query" in intents:
        blocks.append(
            "### Scheme Support Steps\n"
            "1. Check the eligibility conditions in the retrieved scheme text.\n"
            "2. Prepare identity, income, and residence documents if required.\n"
            "3. Use only the official application route mentioned in the source.\n"
            "4. Save the scheme name, office/portal, and application reference number."
        )

    if "legal_summarization" in intents:
        blocks.append(
            "### Reading Help\n"
            "Use the summary first, then open the cited section for the exact wording."
        )

    if not blocks:
        blocks.append("### Citizen Action Pack\nNo extra action steps were generated for this query.")

    return "\n\n".join(blocks)

In [0]:
def legal_rag_answer(query: str) -> Dict[str, object]:
    chunks = retrieve_legal(query, k=5)
    answer = generate_grounded_answer(query, chunks, "legal_qa")
    return {"answer": answer, "sources": chunks}

def scheme_rag_answer(query: str) -> Dict[str, object]:
    chunks = retrieve_scheme(query, k=5)
    answer = generate_grounded_answer(query, chunks, "scheme_query")
    return {"answer": answer, "sources": chunks}

def ipc_bns_rag_answer(query: str) -> Dict[str, object]:
    chunks = retrieve_ipc_bns(query, k=6)
    answer = generate_grounded_answer(query, chunks, "ipc_bns_comparison")
    return {"answer": answer, "sources": chunks}

def summary_rag_answer(query: str) -> Dict[str, object]:
    chunks = retrieve_summary(query, k=5)
    answer = generate_grounded_answer(query, chunks, "legal_summarization")
    return {"answer": answer, "sources": chunks}

In [0]:
def format_final_response(answer_parts: Dict[str, str], action_pack: str, sources: List[Dict]) -> str:
    blocks = []

    if answer_parts.get("legal_qa"):
        blocks.append("## Legal Answer\n" + answer_parts["legal_qa"])

    if answer_parts.get("scheme_query"):
        blocks.append("## Scheme Match\n" + answer_parts["scheme_query"])

    if answer_parts.get("ipc_bns_comparison"):
        blocks.append("## IPC → BNS Comparison\n" + answer_parts["ipc_bns_comparison"])

    if answer_parts.get("legal_summarization"):
        blocks.append("## Summary\n" + answer_parts["legal_summarization"])

    if action_pack:
        blocks.append(action_pack)

    if sources:
        src_lines = []
        for i, s in enumerate(dedupe_sources(sources), start=1):
            src_lines.append(
                f"{i}. {s.get('title', 'Unknown')} | page {s.get('page', 'NA')} | "
                f"{s.get('source_file', 'NA')}"
            )
        blocks.append("### Sources\n" + "\n".join(src_lines))

    return "\n\n".join(blocks)

In [0]:
def run_rag_pipeline(query: str, intents: List[str]) -> Dict[str, object]:
    answer_parts = {}
    all_sources = []

    if "legal_qa" in intents:
        result = legal_rag_answer(query)
        answer_parts["legal_qa"] = result["answer"]
        all_sources.extend(result["sources"])

    if "scheme_query" in intents:
        result = scheme_rag_answer(query)
        answer_parts["scheme_query"] = result["answer"]
        all_sources.extend(result["sources"])

    if "ipc_bns_comparison" in intents:
        result = ipc_bns_rag_answer(query)
        answer_parts["ipc_bns_comparison"] = result["answer"]
        all_sources.extend(result["sources"])

    if "legal_summarization" in intents:
        result = summary_rag_answer(query)
        answer_parts["legal_summarization"] = result["answer"]
        all_sources.extend(result["sources"])

    action_pack = build_citizen_action_pack(intents, query)
    final_answer = format_final_response(answer_parts, action_pack, all_sources)

    return {
        "query": query,
        "intents": intents,
        "answer_parts": answer_parts,
        "sources": dedupe_sources(all_sources),
        "action_pack": action_pack,
        "final_answer": final_answer,
    }

In [0]:
test_query = "I was cheated in an online payment, what law applies and is there any government scheme for help?"

test_intents = ["legal_qa", "scheme_query"]

result = run_rag_pipeline(test_query, test_intents)
print(result["final_answer"])

In [0]:
from typing import TypedDict

class AgentState(TypedDict):
    query: str
    intents: List[str]
    legal_answer: str
    scheme_answer: str
    comparison_answer: str
    summary_answer: str
    final_answer: str

def legal_rag_node(state: AgentState):
    res = legal_rag_answer(state["query"])
    return {"legal_answer": res["answer"]}

def scheme_rag_node(state: AgentState):
    res = scheme_rag_answer(state["query"])
    return {"scheme_answer": res["answer"]}

def comparison_node(state: AgentState):
    res = ipc_bns_rag_answer(state["query"])
    return {"comparison_answer": res["answer"]}

def summarization_node(state: AgentState):
    res = summary_rag_answer(state["query"])
    return {"summary_answer": res["answer"]}

def combine_answers(state: AgentState):
    outputs = []

    if state.get("legal_answer"):
        outputs.append("## Legal Answer\n" + state["legal_answer"])

    if state.get("scheme_answer"):
        outputs.append("## Scheme Match\n" + state["scheme_answer"])

    if state.get("comparison_answer"):
        outputs.append("## IPC → BNS Comparison\n" + state["comparison_answer"])

    if state.get("summary_answer"):
        outputs.append("## Summary\n" + state["summary_answer"])

    return {"final_answer": "\n\n".join(outputs)}

In [0]:

# ============================================================
# Legal RAG Retrieval Pipeline
# ============================================================

from databricks.vector_search.client import VectorSearchClient
from databricks.sdk import WorkspaceClient

# ------------------------------------------------------------
# Configuration
# ------------------------------------------------------------

VECTOR_SEARCH_ENDPOINT = "legal_rag_endpoint"
VECTOR_INDEX_NAME = "workspace.default.legal_docs_vector_index"

RETURN_COLUMNS = [
    "chunk_id",
    "title",
    "page",
    "source_file",
    "domain",
    "text"
]

EMBEDDING_MODEL = "databricks-bge-large-en"

TOP_K = 5


# ------------------------------------------------------------
# Initialize Clients
# ------------------------------------------------------------

vsc = VectorSearchClient()
index = vsc.get_index(
    endpoint_name=VECTOR_SEARCH_ENDPOINT,
    index_name=VECTOR_INDEX_NAME
)

w = WorkspaceClient()


# ------------------------------------------------------------
# Vector Search Retrieval
# ------------------------------------------------------------

def retrieve_chunks(query: str, domain_filter=None, top_k=TOP_K):

    filters = None

    if domain_filter is not None:
        filters = {"domain": domain_filter}

    results = index.similarity_search(
        query_text=query,
        columns=RETURN_COLUMNS,
        num_results=top_k,
        filters=filters
    )

    docs = []

    rows = results["result"]["data_array"]

    for r in rows:
        doc = {
            "chunk_id": r[0],
            "title": r[1],
            "page": r[2],
            "source_file": r[3],
            "domain": r[4],
            "text": r[5]
        }

        docs.append(doc)

    return docs


# ------------------------------------------------------------
# Prompt Builder
# ------------------------------------------------------------

def build_prompt(query, docs):

    context = "\n\n".join([d["text"] for d in docs])

    prompt = f"""
You are a legal assistant helping Indian citizens understand laws.

Answer ONLY using the context provided.

If the answer cannot be found in the context, say:
"I could not find this information in the legal documents."

Context:
{context}

User Question:
{query}

Provide:
1. Clear explanation
2. Relevant law if available
3. Steps the citizen should take
"""

    return prompt


# ------------------------------------------------------------
# LLM Call
# ------------------------------------------------------------

def generate_answer(prompt):

    response = w.serving_endpoints.query(
        name="databricks-dbrx-instruct",
        inputs={
            "prompt": prompt,
            "max_tokens": 400
        }
    )

    return response.predictions[0]


# ------------------------------------------------------------
# RAG Nodes
# ------------------------------------------------------------

def legal_rag_answer(query):

    docs = retrieve_chunks(query, domain_filter="legal")

    if not docs:
        return "No legal documents found.", [], ""

    prompt = build_prompt(query, docs)

    answer = generate_answer(prompt)

    sources = [d["source_file"] for d in docs]

    return answer, docs, sources


def scheme_rag_answer(query):

    docs = retrieve_chunks(query, domain_filter="scheme")

    if not docs:
        return "No government schemes found.", [], ""

    prompt = build_prompt(query, docs)

    answer = generate_answer(prompt)

    sources = [d["source_file"] for d in docs]

    return answer, docs, sources


def ipc_bns_rag_answer(query):

    docs = retrieve_chunks(query, domain_filter="ipc_bns")

    if not docs:
        return "No IPC/BNS comparison found.", [], ""

    prompt = build_prompt(query, docs)

    answer = generate_answer(prompt)

    sources = [d["source_file"] for d in docs]

    return answer, docs, sources


def summary_rag_answer(query):

    docs = retrieve_chunks(query, domain_filter="summary")

    if not docs:
        return "No summary documents found.", [], ""

    prompt = build_prompt(query, docs)

    answer = generate_answer(prompt)

    sources = [d["source_file"] for d in docs]

    return answer, docs, sources


# ------------------------------------------------------------
# Final RAG Pipeline
# ------------------------------------------------------------

def run_rag_pipeline(query, intents):

    answer_parts = {}
    all_sources = []

    # -----------------------------------
    # Legal Retrieval
    # -----------------------------------

    if "legal_advice" in intents:

        ans, docs, src = legal_rag_answer(query)

        answer_parts["legal"] = ans
        all_sources.extend(src)

    # -----------------------------------
    # Government Schemes
    # -----------------------------------

    if "government_scheme" in intents:

        ans, docs, src = scheme_rag_answer(query)

        answer_parts["scheme"] = ans
        all_sources.extend(src)

    # -----------------------------------
    # IPC vs BNS
    # -----------------------------------

    if "ipc_bns_compare" in intents:

        ans, docs, src = ipc_bns_rag_answer(query)

        answer_parts["ipc_bns"] = ans
        all_sources.extend(src)

    # -----------------------------------
    # Summary
    # -----------------------------------

    if "summary" in intents:

        ans, docs, src = summary_rag_answer(query)

        answer_parts["summary"] = ans
        all_sources.extend(src)

    # -----------------------------------
    # Combine Answers
    # -----------------------------------

    final_answer = "\n\n".join(answer_parts.values())

    return {
        "final_answer": final_answer,
        "answer_parts": answer_parts,
        "sources": list(set(all_sources)),
        "action_pack": "Visit your nearest legal aid center or consult an advocate if needed."
    }